# Cell 1 — Setup + find swapped dataset

In [1]:
!pip install -q ultralytics huggingface_hub scikit-learn opencv-python pyyaml

from pathlib import Path
import shutil
import yaml
import pandas as pd
import numpy as np
import torch
import cv2
from collections import Counter, defaultdict
from ultralytics import YOLO
from huggingface_hub import hf_hub_download

print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

INPUT_ROOT = Path("input")
WORK_DIR = Path("/kaggle/working")

# Find YOLO dataset
candidate_dirs = []

for yaml_path in INPUT_ROOT.rglob("data.yaml"):
    folder = yaml_path.parent
    
    if not (folder / "images" / "train").exists():
        continue
    if not (folder / "labels" / "train").exists():
        continue
    if not (folder / "images" / "val").exists():
        continue
    if not (folder / "labels" / "val").exists():
        continue
    if not (folder / "images" / "test").exists():
        continue
    if not (folder / "labels" / "test").exists():
        continue

    score = 0
    names_list = []

    try:
        cfg = yaml.safe_load(yaml_path.read_text())
        names = cfg.get("names", [])
        names_list = list(names.values()) if isinstance(names, dict) else list(names)
        joined = " ".join([str(x).lower() for x in names_list])

        if "player_team1" in joined:
            score += 10
        if "player_team2" in joined:
            score += 10
        if "ball" in joined:
            score += 5
        if "referee" in joined:
            score += 5
        if "black" in str(folder).lower() or "swapped" in str(folder).lower():
            score += 3

    except Exception:
        pass

    candidate_dirs.append((score, folder, names_list))

if len(candidate_dirs) == 0:
    raise FileNotFoundError("Could not find YOLO dataset in /kaggle/input")

candidate_dirs = sorted(candidate_dirs, key=lambda x: x[0], reverse=True)

original_dataset_dir = candidate_dirs[0][1]

print("\nDatasets found:")
for i, (score, folder, names) in enumerate(candidate_dirs):
    print(i, "| score:", score, "|", folder, "|", names)

print("\nUsing dataset:", original_dataset_dir)

# =========================================================
# Robust helpers for Kaggle restarts / reruns
# =========================================================

def find_model_path(prefer="best", raise_error=True):
    """
    Locate the trained YOLO model in /kaggle/working.
    Prefer best.pt, then last.pt. This is safer than relying only on
    notebook variables, which disappear after a Kaggle/kernel restart.
    """
    search_roots = []
    if "PROJECT_DIR" in globals():
        search_roots.append(Path(PROJECT_DIR))
    search_roots.append(WORK_DIR)

    variable_candidates = []
    for var_name in ["BEST_MODEL_PATH", "LAST_MODEL_PATH"] if prefer == "best" else ["LAST_MODEL_PATH", "BEST_MODEL_PATH"]:
        if var_name in globals():
            try:
                p = Path(globals()[var_name])
                if p.exists():
                    variable_candidates.append(p)
            except Exception:
                pass

    pattern_order = ["**/weights/best.pt", "**/best.pt", "**/weights/last.pt", "**/last.pt"]
    if prefer == "last":
        pattern_order = ["**/weights/last.pt", "**/last.pt", "**/weights/best.pt", "**/best.pt"]

    candidates = list(variable_candidates)
    for root in search_roots:
        if root.exists():
            for pattern in pattern_order:
                candidates.extend(root.rglob(pattern))

    # Keep unique existing candidates and prefer newest file.
    unique = []
    seen = set()
    for p in candidates:
        p = Path(p)
        if p.exists() and p not in seen:
            unique.append(p)
            seen.add(p)

    if unique:
        unique = sorted(unique, key=lambda p: p.stat().st_mtime, reverse=True)
        return unique[0]

    if raise_error:
        raise RuntimeError(
            "No trained model was found in /kaggle/working. "
            "This usually means the Kaggle session/kernel restarted after being idle, "
            "so temporary outputs were deleted. Run the notebook from the top in one session, "
            "or rerun the training cell before validation/export."
        )
    return None


def safe_export_outputs(stage="final", cleanup_heavy=False, require_model=False):
    """
    Create a small zip with the model, CSV results, and important plots.
    Also works as a checkpoint export after training/validation so results
    are available even if a later analysis cell fails.
    """
    export_name = "final_swapped_3class_outputs" if stage == "final" else f"checkpoint_{stage}_swapped_3class_outputs"
    final_dir = WORK_DIR / export_name

    if final_dir.exists():
        shutil.rmtree(final_dir)
    final_dir.mkdir(parents=True, exist_ok=True)

    print(f"Creating export folder: {final_dir}")

    copied_any = False
    copied_model = False

    # Copy best/last model if present.
    best_path = find_model_path("best", raise_error=False)
    last_path = find_model_path("last", raise_error=False)

    if best_path is not None:
        shutil.copy2(best_path, final_dir / "best.pt")
        print("Copied best model:", best_path)
        copied_any = True
        copied_model = True

    if last_path is not None and last_path != best_path:
        shutil.copy2(last_path, final_dir / "last.pt")
        print("Copied last model:", last_path)
        copied_any = True
        copied_model = True

    if require_model and not copied_model:
        raise RuntimeError(
            "Could not export because no best.pt or last.pt exists in /kaggle/working. "
            "Your Kaggle session most likely restarted or the training cell did not finish. "
            "Run all cells from the top without leaving the session idle, then run this export cell."
        )

    # Copy CSV results from anywhere under /kaggle/working.
    csv_patterns = [
        "three_class_swapped_per_class.csv",
        "three_class_swapped_summary.csv",
        "swapped_error_analysis_summary.csv",
        "swapped_threshold_comparison.csv",
        "swapped_image_error_analysis.csv",
    ]

    for pattern in csv_patterns:
        for p in WORK_DIR.rglob(pattern):
            if final_dir in p.parents:
                continue
            target = final_dir / p.name
            shutil.copy2(p, target)
            print("Copied CSV:", p)
            copied_any = True

    # Copy important training/evaluation plots only.
    important_plot_names = {
        "results.png",
        "confusion_matrix.png",
        "confusion_matrix_normalized.png",
        "F1_curve.png",
        "P_curve.png",
        "R_curve.png",
        "PR_curve.png",
    }

    for p in WORK_DIR.rglob("*.png"):
        if final_dir in p.parents:
            continue
        if p.name in important_plot_names:
            target_name = p.parent.name + "_" + p.name
            shutil.copy2(p, final_dir / target_name)
            print("Copied plot:", p)
            copied_any = True

    readme_text = f"""
YOLOv8 3-class football model outputs
Export stage: {stage}

Classes:
0 = player
1 = ball
2 = referee

Expected important files:
- best.pt / last.pt
- three_class_swapped_per_class.csv
- three_class_swapped_summary.csv
- swapped_error_analysis_summary.csv
- swapped_threshold_comparison.csv
- swapped_image_error_analysis.csv

Note:
If model files are missing, Kaggle probably restarted the session and deleted /kaggle/working.
Run all cells from the top in one session, or rerun training before export.
"""
    (final_dir / "README.txt").write_text(readme_text)

    zip_path = shutil.make_archive(
        base_name=str(WORK_DIR / export_name),
        format="zip",
        root_dir=str(final_dir)
    )

    print("Created zip:", zip_path)

    if cleanup_heavy:
        heavy_paths = [
            WORK_DIR / "football_yolo_dataset_3cls_swapped",
            WORK_DIR / "three_class_swapped_experiments",
            WORK_DIR / "runs",
        ]

        for path in heavy_paths:
            if path.exists():
                print("Deleting heavy folder:", path)
                shutil.rmtree(path, ignore_errors=True)

        for cache_file in WORK_DIR.rglob("*.cache"):
            try:
                cache_file.unlink()
                print("Deleted cache:", cache_file)
            except Exception:
                pass

        gc.collect()

    print("\nFiles in export folder:")
    for p in sorted(final_dir.iterdir()):
        size_mb = p.stat().st_size / (1024 * 1024)
        print(f"{p.name} - {size_mb:.2f} MB")

    if not copied_any:
        print("\nWARNING: No model/results were found. This zip only contains README.txt.")

    return zip_path




Torch: 2.6.0+cu126
CUDA: True
GPU: NVIDIA GeForce RTX 4060 Ti

Datasets found:
0 | score: 30 | input | ['player_team1', 'player_team2', 'ball', 'referee']

Using dataset: input


# Cell 2 — Verify split

In [2]:
import re
from collections import Counter

def get_video_name(stem):
    if "_frame_" in stem:
        return stem.split("_frame_")[0]
    return "unknown"

def extract_frame_number(path):
    match = re.search(r"_frame_(\d+)", path.stem)
    if match:
        return int(match.group(1))
    return -1

def count_videos(images_dir):
    counts = Counter()
    frames = defaultdict(list)

    for p in images_dir.glob("*.jpg"):
        video = get_video_name(p.stem)
        frame = extract_frame_number(p)
        counts[video] += 1
        frames[video].append(frame)

    return counts, frames

train_counts, train_frames = count_videos(original_dataset_dir / "images" / "train")
val_counts, val_frames = count_videos(original_dataset_dir / "images" / "val")
test_counts, test_frames = count_videos(original_dataset_dir / "images" / "test")

print("TRAIN videos:")
for video, count in train_counts.most_common():
    fs = train_frames[video]
    print(video, "| count:", count, "| min:", min(fs), "| max:", max(fs))

print("\nVAL videos:")
for video, count in val_counts.most_common():
    fs = val_frames[video]
    print(video, "| count:", count, "| min:", min(fs), "| max:", max(fs))

print("\nTEST videos:")
for video, count in test_counts.most_common():
    fs = test_frames[video]
    print(video, "| count:", count, "| min:", min(fs), "| max:", max(fs))

def count_label_classes(labels_dir):
    counter = Counter()

    for label_path in labels_dir.glob("*.txt"):
        for line in label_path.read_text().splitlines():
            parts = line.strip().split()
            if len(parts) >= 1:
                counter[int(float(parts[0]))] += 1

    return counter

print("\nOriginal TRAIN label counts:", count_label_classes(original_dataset_dir / "labels" / "train"))
print("Original VAL label counts:", count_label_classes(original_dataset_dir / "labels" / "val"))
print("Original TEST label counts:", count_label_classes(original_dataset_dir / "labels" / "test"))

TRAIN videos:
lightblue_green_white | count: 1057 | min: 0 | max: 5280
Green_and_Blue | count: 720 | min: 0 | max: 3595
hashemmatch02 | count: 720 | min: 0 | max: 3595
lightblue_red | count: 720 | min: 0 | max: 3595
black_and_red | count: 703 | min: 180 | max: 3690
Green_and_Blue_part_2 | count: 360 | min: 0 | max: 1795

VAL videos:
white_and_black_2_mins | count: 742 | min: 0 | max: 3705
hashemmatch01 | count: 720 | min: 0 | max: 3595

TEST videos:
white_and_yellow | count: 720 | min: 0 | max: 3595

Original TRAIN label counts: Counter({1: 20531, 0: 19879, 3: 4716, 2: 3538})
Original VAL label counts: Counter({1: 8472, 0: 7665, 3: 1614, 2: 1414})
Original TEST label counts: Counter({0: 4619, 1: 4486, 3: 996, 2: 645})


# Cell 3 — Convert 4 classes

In [3]:
# =========================================================
# Convert 4-class dataset to 3-class dataset
# =========================================================

CONVERTED_DIR = WORK_DIR / "football_yolo_dataset_3cls_swapped"

if CONVERTED_DIR.exists():
    shutil.rmtree(CONVERTED_DIR)

for split in ["train", "val", "test"]:
    (CONVERTED_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
    (CONVERTED_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)

# Old classes:
# 0 player_team1 -> 2 player
# 1 player_team2 -> 2 player
# 2 ball         -> 0 ball
# 3 referee      -> 3 referee

old_to_new = {
    0: 2,
    1: 2,
    2: 0,
    3: 3
}

def convert_label_file(src_label_path, dst_label_path):
    new_lines = []

    if not src_label_path.exists():
        dst_label_path.write_text("")
        return Counter()

    class_counter = Counter()

    for line in src_label_path.read_text().splitlines():
        parts = line.strip().split()

        if len(parts) != 5:
            continue

        old_cls = int(float(parts[0]))

        if old_cls not in old_to_new:
            continue

        new_cls = old_to_new[old_cls]
        new_line = " ".join([str(new_cls)] + parts[1:])
        new_lines.append(new_line)

        class_counter[new_cls] += 1

    dst_label_path.write_text("\n".join(new_lines))

    return class_counter

overall_counts = {
    "train": Counter(),
    "val": Counter(),
    "test": Counter()
}

for split in ["train", "val", "test"]:
    src_img_dir = original_dataset_dir / "images" / split
    src_lbl_dir = original_dataset_dir / "labels" / split

    dst_img_dir = CONVERTED_DIR / "images" / split
    dst_lbl_dir = CONVERTED_DIR / "labels" / split

    image_paths = sorted(list(src_img_dir.glob("*.jpg")))

    print(f"\nConverting {split}: {len(image_paths)} images")

    for img_path in image_paths:
        shutil.copy2(img_path, dst_img_dir / img_path.name)

        src_label_path = src_lbl_dir / f"{img_path.stem}.txt"
        dst_label_path = dst_lbl_dir / f"{img_path.stem}.txt"

        counts = convert_label_file(src_label_path, dst_label_path)
        overall_counts[split].update(counts)

print("\nConverted label counts:")
for split in ["train", "val", "test"]:
    print(split, overall_counts[split])

data_yaml_3cls = CONVERTED_DIR / "data.yaml"

data_cfg = {
    "path": str(CONVERTED_DIR),
    "train": "train_weighted.txt",
    "val": "images/val",
    "test": "images/test",
    "names": {
        0: "player",
        1: "ball",
        2: "referee"
    }
}

with open(data_yaml_3cls, "w") as f:
    yaml.safe_dump(data_cfg, f, sort_keys=False)

print("\nCreated:", data_yaml_3cls)
print("Converted dataset:", CONVERTED_DIR)


Converting train: 4280 images

Converting val: 1462 images

Converting test: 720 images

Converted label counts:
train Counter({2: 40410, 3: 4716, 0: 3538})
val Counter({2: 16137, 3: 1614, 0: 1414})
test Counter({2: 9105, 3: 996, 0: 645})

Created: \kaggle\working\football_yolo_dataset_3cls_swapped\data.yaml
Converted dataset: \kaggle\working\football_yolo_dataset_3cls_swapped


# Cell 4 — YOLO validation on test

In [4]:
from pathlib import Path

PROJECT_DIR = Path("validation_results")
PROJECT_DIR.mkdir(exist_ok=True)

exp_name = "hf_base_model"

# =========================================================
# Validate best model on test set
# =========================================================

from huggingface_hub import hf_hub_download
from ultralytics import YOLO

hf_model_path = hf_hub_download(
    repo_id="uisikdag/yolo-v8-football-players-detection",
    filename="best.pt"
)

print("Using model:", hf_model_path)

best_model = YOLO(hf_model_path)

metrics = best_model.val(
    data=str(data_yaml_3cls),
    split="test",
    imgsz=960,
    batch=8,
    device=0,
    conf=0.001,
    iou=0.6,
    plots=True,
    save_json=False,
    project=str(PROJECT_DIR),
    name=f"{exp_name}_test_eval"
)

names = ["player", "ball", "referee"]

per_class_rows = []

for i, class_name in enumerate(names):
    per_class_rows.append({
        "experiment": exp_name,
        "class": class_name,
        "precision": float(metrics.box.p[i]) if i < len(metrics.box.p) else None,
        "recall": float(metrics.box.r[i]) if i < len(metrics.box.r) else None,
        "mAP50": float(metrics.box.ap50[i]) if i < len(metrics.box.ap50) else None,
        "mAP50_95": float(metrics.box.ap[i]) if i < len(metrics.box.ap) else None,
    })

per_class_df = pd.DataFrame(per_class_rows)

summary_df = pd.DataFrame([{
    "experiment": exp_name,
    "model": "uisikdag/yolo-v8-football-players-detection",
    "dataset": str(CONVERTED_DIR),
    "imgsz": 960,
    "epochs": 0,
    "batch": 8,
    "mAP50": float(metrics.box.map50),
    "mAP50_95": float(metrics.box.map),
    "precision_mean": float(np.mean(metrics.box.p)),
    "recall_mean": float(np.mean(metrics.box.r)),
}])

per_class_path = WORK_DIR / "three_class_swapped_per_class.csv"
summary_path = WORK_DIR / "three_class_swapped_summary.csv"

per_class_df.to_csv(per_class_path, index=False)
summary_df.to_csv(summary_path, index=False)

print("Summary:")
display(summary_df)

print("Per-class:")
display(per_class_df)

print("Saved:")
print(per_class_path)
print(summary_path)



Using model: C:\Users\USER\.cache\huggingface\hub\models--uisikdag--yolo-v8-football-players-detection\snapshots\01c4d0e18813ac75a2c73cc35145bc240af85342\best.pt
Ultralytics 8.3.213  Python-3.12.9 torch-2.6.0+cu126 CUDA:0 (NVIDIA GeForce RTX 4060 Ti, 8188MiB)
Model summary (fused): 72 layers, 11,127,132 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 8.71.8 MB/s, size: 58.5 KB)
val: Scanning C:\kaggle\working\football_yolo_dataset_3cls_swapped\labels\test... 720 images, 18 backgrounds, 619 corrupt: 100% ━━━━━━━━━━━━ 720/720 406.1it/s 1.8s0.1s
val: C:\kaggle\working\football_yolo_dataset_3cls_swapped\images\test\white_and_yellow_frame_000000.jpg: ignoring corrupt image/label: Label class 3 exceeds dataset class count 3. Possible class labels are 0-2
val: C:\kaggle\working\football_yolo_dataset_3cls_swapped\images\test\white_and_yellow_frame_000005.jpg: ignoring corrupt image/label: Label class 3 exceeds dataset class count 3. Possible class labels ar

,experiment,model,dataset,imgsz,epochs,batch,mAP50,mAP50_95,precision_mean,recall_mean
0,hf_base_model,uisikdag/yolo-v8-football-players-detection,\kaggle\working\football_yolo_dataset_3cls_swa...,960,0,8,0.332294,0.095962,0.610692,0.329752


Per-class:


,experiment,class,precision,recall,mAP50,mAP50_95
0,hf_base_model,player,0.896810,0.291667,0.420755,0.138718
1,hf_base_model,ball,0.324575,0.367837,0.243834,0.053207
2,hf_base_model,referee,NaN,NaN,NaN,NaN


Saved:
\kaggle\working\three_class_swapped_per_class.csv
\kaggle\working\three_class_swapped_summary.csv
